In [ ]:
from import_ipynb import NotebookFinder  # type: ignore
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score
import importlib
import os
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import numpy as np


In [ ]:
# retrouver les dossiers
root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
utils_dir = os.path.join(root, "pneumonia_knn", "notebooks", "utils")

# --- pneumonia_knn\notebooks\utils
spec_print = NotebookFinder().find_spec("prints", [utils_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

In [ ]:
# 1. Définir les métriques pour GridSearchCV
scoring = {
    'accuracy':  'accuracy',
    'precision': make_scorer(precision_score, average='weighted'),
    'recall':    make_scorer(recall_score,    average='macro'),
    'f1':        make_scorer(f1_score,        average='weighted'),
    'roc_auc':   make_scorer(roc_auc_score,   response_method="predict_proba", average='macro', multi_class='ovo')
}
def get_scoring():
    return scoring
# 2. Colonnes à afficher dans les résultats
cols = [
    'params',
    'mean_test_accuracy',
    'mean_test_precision',
    'mean_test_recall',
    'mean_test_f1',
    'mean_test_roc_auc',
    'rank_test_accuracy'
]

In [ ]:
def show_image(pca):
    fig, axes = plt.subplots(1, 5, figsize=(14, 3))

    for i, ax in enumerate(axes):
        composante = pca.components_[i].reshape(224, 224, 3)  # ← 3 canaux
        composante = (composante - composante.min()) / (composante.max() - composante.min())
        
        ax.imshow(composante)  # ← pas cmap='gray'
        ax.set_title(f'PC{i+1} — {pca.explained_variance_ratio_[i]*100:.1f}%', fontsize=9)
        ax.axis('off')

    plt.suptitle("Ce que chaque composante PCA a capturé", fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
def show_all_types_of_metrics_from_hyperparameters_tuning(results, title):
    print(title)
    for result in sorted(results.columns):
        if result.startswith('mean'):
            prints.yellow(result)
        else:
            print(result)

In [ ]:
def show_all_metrics_hyperparameters_tuning(results, title):
    print(title)
    display(results[cols].sort_values('rank_test_accuracy'))

In [ ]:
def show_plot_scatter_prediction(y_test, y_pred, X_reduced, implementation_name):
    class_names = ['NORMAL', 'BACTERIA', 'VIRUS']
    colors      = ['#378ADD', '#22a05a', '#EF9F27']
    
    correct = (y_test == y_pred)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Projection {implementation_name} — KNN predictions", fontsize=14)
    
    # --- Onglet 1 : Vraie classe ---
    ax = axes[0]
    for c, (name, color) in enumerate(zip(class_names, colors)):
        mask = y_test == c
        ax.scatter(X_reduced[mask, 0], X_reduced[mask, 1],
                   c=color, label=name, alpha=0.6, s=15)
    ax.set_title("Vraie classe")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    # --- Onglet 2 : Prédiction KNN ---
    ax = axes[1]
    for c, (name, color) in enumerate(zip(class_names, colors)):
        mask = y_pred == c
        ax.scatter(X_reduced[mask, 0], X_reduced[mask, 1],
                   c=color, label=f"Prédit {name}", alpha=0.6, s=15)
    ax.set_title("Prédiction KNN")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    # --- Onglet 3 : Erreurs ---
    ax = axes[2]
    ax.scatter(X_reduced[correct, 0],  X_reduced[correct, 1],
               c='#cccccc', alpha=0.3, s=12, label='Correct')
    ax.scatter(X_reduced[~correct, 0], X_reduced[~correct, 1],
               c='#e24b4a', alpha=0.8, s=20, label='Erreur')
    ax.set_title("Erreurs de prédiction")
    ax.set_xlabel(f"{implementation_name}1")
    ax.set_ylabel(f"{implementation_name}2")
    ax.legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def show_plot_prediction(y_test, y_pred, title = "Métriques de prédiction", y_proba = None):
    metrics = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall':    recall_score(y_test, y_pred, average='weighted'),
        'F1':        f1_score(y_test, y_pred, average='weighted'),
    }
    
    if y_proba is not None:
        metrics['ROC AUC'] = roc_auc_score(y_test, y_proba, average='macro', multi_class='ovo')

    fig, ax = plt.subplots(figsize=(14, 6))  # ← plus large et plus haut
    
    bars = ax.barh(
        list(metrics.keys()), 
        list(metrics.values()), 
        color=['#378ADD', '#22a05a', '#EF9F27', '#e24b4a', '#9b59b6'],  # une couleur par barre
        alpha=0.8
    )    
    for bar, val in zip(bars, metrics.values()):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)  # ← police plus petite
    
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Score", fontsize=9)        # ← plus petit
    ax.set_title(title, fontsize=11)          # ← plus petit
    ax.tick_params(axis='both', labelsize=8)  # ← étiquettes axes plus petites
    ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_true_positives(y_test, y_pred, title=""):
    classes = ['NORMAL', 'BACTERIA', 'VIRUS']
    cm = confusion_matrix(y_test, y_pred)
    
    # Les vrais positifs sont sur la diagonale
    tp = cm.diagonal()
    total = cm.sum(axis=1)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    
    x = np.arange(len(classes))
    bars_tp    = ax.bar(x - 0.2, tp,    width=0.4, label='Vrais positifs', color='#22a05a', alpha=0.8)
    bars_total = ax.bar(x + 0.2, total, width=0.4, label='Total réel',     color='#378ADD', alpha=0.8)
    
    # Afficher les valeurs sur les barres
    for bar in bars_tp:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', fontsize=9)
    for bar in bars_total:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', fontsize=9)
    
    ax.set_xticks(x)
    ax.set_xticklabels(classes, fontsize=10)
    ax.set_ylabel("Nombre d'images", fontsize=9)
    ax.set_title(f"Vrais positifs par classe — {title}", fontsize=11)
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()
